In [20]:
pip install -qq langchain langchain-openai langchain-community chromadb sentence-transformers openai pypdf tiktoken dotenv

Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
from dotenv import load_dotenv
                                                 
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")

# Datasource

In [22]:
import urllib.request

url = "https://isap.sejm.gov.pl/isap.nsf/download.xsp/WDU19970780483/U/D19970483Lj.pdf"
urllib.request.urlretrieve(url, "constitution.pdf")

('constitution.pdf', <http.client.HTTPMessage at 0x137867950>)

In [23]:
from pypdf import PdfReader

reader = PdfReader("constitution.pdf")
raw_text = "\n".join(page.extract_text() for page in reader.pages)
print(len(raw_text))

108868


In [24]:
import re

# Based on document structure, find best split for chunks
chunks = re.split(r'(Art\. \d+\.)', raw_text)

# Add source to chunk
articles = []
for i in range(1, len(chunks), 2):
    header = chunks[i].strip()
    body = chunks[i+1].strip() if i+1 < len(chunks) else ""
    articles.append({"text": f"{header} {body}", "source": header})

In [25]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

docs = [Document(page_content=a["text"], metadata={"source": a["source"]}) for a in articles]
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=API_KEY)

vectorstore = Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db2")

# Question pipeline

In [26]:
from langchain.agents import create_agent
from langchain.tools import tool

In [27]:
@tool(response_format="content_and_artifact")
def retrieve_constitution_content(query: str):
    """
    Searches for information contained in the Constitution of the Republic of Poland.

    Args:
        query: Question or phrase to search for

    Returns:
        Tuple containing the formatted answer and the original documents
    """
    retrieved_docs = vectorstore.similarity_search(query, k=1)
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n"
        f"Content: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs



In [28]:
from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-4o-mini"

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, temperature=0.0)

In [29]:
rag_agent = create_agent(
    model=llm,
    tools=[retrieve_constitution_content],
    system_prompt=(
        "You are an expert on the Constitution of the Republic of Poland."
        "Answer EXCLUSIVELY based on the provided context."
        "Always provide the article number."
        "If you cannot find the answer — say"
        "'I did not find the answer in the Constitution.'"      
    ),
)

In [30]:
import textwrap

def ask_lawyer(q):
    result = rag_agent.invoke({
        "messages": [{"role": "user", "content": q}]
    })
    answer = result["messages"][-1].content
    print(f"Question: {q}")
    print("Answer:", textwrap.fill(answer, width=80))
    print("-" * 80)

In [31]:
# constitution specific questions
ask_lawyer("Who appoints the Prime Minister?")
ask_lawyer("How long is the term of the Sejm?")
ask_lawyer("When can a state of emergency be introduced?")

Question: Who appoints the Prime Minister?
Answer: The Prime Minister is appointed by the President of the Republic of Poland, as
stated in Article 154.
--------------------------------------------------------------------------------
Question: How long is the term of the Sejm?
Answer: I did not find the answer in the Constitution.
--------------------------------------------------------------------------------
Question: When can a state of emergency be introduced?
Answer: A state of emergency can be introduced in the case of a threat to the
constitutional order of the state, the safety of citizens, or public order.
According to Article 230, the President of the Republic of Poland may introduce
a state of emergency for a specified period, not exceeding 90 days, upon the
request of the Council of Ministers. The state of emergency can be extended only
once, with the consent of the Sejm, for a period not exceeding 60 days.
-------------------------------------------------------------------

In [32]:
# refuse to hallucinate
ask_lawyer("What is the recipe for butter chicken?")
ask_lawyer("How does the President prepare pancakes according to the Constitution?")

Question: What is the recipe for butter chicken?
Answer: I did not find the answer in the Constitution.
--------------------------------------------------------------------------------
Question: How does the President prepare pancakes according to the Constitution?
Answer: I did not find the answer in the Constitution.
--------------------------------------------------------------------------------
